In [5]:
import pandas as pd
import numpy as np

In [6]:
key = pd.read_csv(r'key.csv')
message = pd.read_csv(r'message.csv')
codebook = pd.read_csv(r'codebook.csv')

In [7]:
# находим выключенную стануцию 
key.groupby('relay')['shift'].count().sort_values()

# ответ Spencer-61  

relay
Spencer-61            0
LaTrappe-42           1
Achel-19            126
Westmalle-31        132
Westvleteren-12     132
Rochefort-10        133
Engelszell-28       133
Zundert-56          134
Koningshoeven-05    135
TreFontane-33       137
TyntMeadow-88       140
LaTrappe-42         140
Chimay-23           141
MontDesCats-77      152
Orval-07            152
Name: shift, dtype: int64

In [8]:
# находим стануцию c одним числом 
print(key.groupby('relay')['shift'].nunique().sort_values())

# узнаем число 
print(key.query("relay == 'MontDesCats-77'")['shift'])

# провеирка
assert key.query("relay == 'MontDesCats-77'")['shift'].nunique() == 1

# ответ MontDesCats-77 число 176 

relay
Spencer-61            0
MontDesCats-77        1
LaTrappe-42           1
Orval-07              4
Achel-19             95
Rochefort-10         99
Westvleteren-12     100
TreFontane-33       101
Zundert-56          101
Westmalle-31        101
Engelszell-28       103
Chimay-23           103
LaTrappe-42         104
TyntMeadow-88       106
Koningshoeven-05    107
Name: shift, dtype: int64
1672    176
1673    176
1674    176
1675    176
1676    176
       ... 
1819    176
1820    176
1821    176
1822    176
1823    176
Name: shift, Length: 152, dtype: str


In [9]:
# ищем станцию которая пиредает сигнали по кругу
print(key.groupby('relay')['shift'].nunique())

# провирка я не знаю зачим пусть будет тренирус писать аля тесты
assert set(key.loc[key['relay'] == 'Orval-07', 'shift'].unique()) <= {'140', '196', '34', '94'}

# key[key['relay'] == 'Orval-07']
# выводим и провиряем что чиселки по кругу
key.loc[key['relay'] == 'Orval-07'].head(12)

# ответ Orval-07 число 4

relay
Achel-19             95
Chimay-23           103
Engelszell-28       103
Koningshoeven-05    107
LaTrappe-42         104
LaTrappe-42           1
MontDesCats-77        1
Orval-07              4
Rochefort-10         99
Spencer-61            0
TreFontane-33       101
TyntMeadow-88       106
Westmalle-31        101
Westvleteren-12     100
Zundert-56          101
Name: shift, dtype: int64


,pos,shift,relay
1368,0,94,Orval-07
1369,1,34,Orval-07
1370,2,196,Orval-07
1371,3,140,Orval-07
1372,4,94,Orval-07
1373,5,34,Orval-07
1374,6,196,Orval-07
1375,7,140,Orval-07
1376,8,94,Orval-07
1377,9,34,Orval-07


In [10]:
# заполняем все nan в 0, а весь мусор в nan
numeric_shift = pd.to_numeric(key.fillna(0)['shift'], errors='coerce')

# создаем маску где nan эта true
trash_mask = numeric_shift.isna()

# смотрим где есть мусор и сколько
trash_counts = key[trash_mask].groupby('relay').size()
trash_counts

# ответ Achel-19 число 26

relay
Achel-19    26
dtype: int64

In [11]:
# оставляем наши 10 станций где есть 2 предателя 
candidates = key.groupby('relay')['shift'].apply(lambda x: x.isna().sum()).sort_values().index[2:12]

# заполняем весь мусор в nan
numeric_shift = pd.to_numeric(key['shift'], errors='coerce')

# создаем маску где nan эта false
without_trash_mask = numeric_shift.notna()

# убирем мусор и лишнии станции 
clean_key = key[without_trash_mask]
clean_key = clean_key[clean_key['relay'].isin(candidates)]

# собираем настоящий ключ, берем моду значения для каждой позиции
true_key = clean_key.groupby('pos')['shift'].apply(lambda x: x.mode().iloc[0]) \
    .reset_index().rename(columns={'shift': 'true_shift'})

# объединяем чистый ключ с нашим датасетом
merged = clean_key.merge(true_key, on='pos', how='left')

# считаем количество замен
merged['substitutions'] = merged['shift'] != merged['true_shift']

# смотрим у кого много замен
merged.groupby('relay')['substitutions'].sum().sort_values(ascending=False)

# ответ Chimay-23, Westmalle-31 

relay
Chimay-23           99
Engelszell-28       22
LaTrappe-42         19
TyntMeadow-88       16
TreFontane-33       15
Zundert-56          15
Koningshoeven-05    13
Westvleteren-12     13
Rochefort-10        12
LaTrappe-42          1
Name: substitutions, dtype: int64

In [19]:
import hashlib

key_list = list(true_key['true_shift'])
my_hash = hashlib.sha256(",".join(map(str, key_list)).encode()).hexdigest()
needed_hash = '08a09df10bbe00b15d9fbe7f0c2937cd3e6986aa920adc7fab83d762c443ecfd'
my_hash == needed_hash
# sum([int(i) for i in key_list])

True

In [25]:
# считаем размер алфавит
alphabet = codebook['char'].nunique()

# письмо с ключом
message_with_key = message.merge(true_key, on='pos', how='left')

# получаем код
true_code = (message_with_key['code'].astype(int) - message_with_key['true_shift'].astype(int)) % alphabet

# переводим в символы
messgae_str = ''
for i in true_code:
    messgae_str += codebook.iloc[i]['char']

print(messgae_str[-12:])
messgae_str

:чUδХAдЙЙ,ъU


'СзYN»С[ччЙчъ{П=»}[ФъrτQФччНIHFHчοПυкЕeчHжчЕ}:~QЛ§§δОΖQ»ЛЛP3гQПγпФЕчCл}8ч~{Q§чбСЭЙ,чUСчο,wρПчC2ч«UQ{ХUХЙrЕHτHЙQCПwВзЛЯИЛчкчUIQγчU§Ж§CHЩЕγ}СчФ:чUδХAдЙЙ,ъU'

In [26]:
message_with_key

,pos,code,true_shift
0,73,170,119
1,12,3,206
2,90,137,23
3,22,45,153
4,62,155,78
...,...,...,...
147,106,34,202
148,65,87,44
149,105,3,146
150,3,201,174


In [27]:
true_key

,pos,true_shift
0,0,130
1,1,121
2,2,83
3,3,174
4,4,77
...,...,...
147,147,74
148,148,80
149,149,41
150,150,66


In [16]:
# honest_relayes = ['LaTrappe-42', 'TyntMeadow-88', 'TreFontane-33',
# 'Koningshoeven-05', 'Zundert-56', 'Engelszell-28', 'Rochefort-10',
# 'Westvleteren-12']

# honest_key = clean_key[clean_key['relay'].isin(honest_relayes)]
# honest_key